# Training launcher

This notebook uses [training_config.json](training_config.json) to configure the training run and the SmartDoc15 dataset paths.

First load/adjust the config, then optionally start training.

In [1]:
from pathlib import Path
import json
import sys

finetune_dir = Path('/workspaces/semedo/src/finetune')
if str(finetune_dir) not in sys.path:
    sys.path.insert(0, str(finetune_dir))

from finetuning import load_training_config

config = load_training_config()
print('Loaded config keys:', sorted(config.keys()))
print('Model path:', config.get('model_path'))
print('SmartDoc root:', config.get('root_folder'))
print('SmartDoc metadata:', config.get('opensource_metadata_path'))
print('UKE metadata:', config.get('uke_metadata_path'))

Loaded config keys: ['active_uke_fold_index', 'allow_duplicate_samples_per_epoch', 'backgrounds', 'balance_train_domains', 'base_run_name', 'checkpoint_dir', 'debug_quick_uke_run', 'epochs', 'include_uke_in_training', 'lr_list', 'model_path', 'n_test_per_subfolder', 'opensource_image_folder_path', 'opensource_metadata_path', 'patience', 'ratio_scenarios', 'root_folder', 'run_5_repeat_comparison', 'run_all_ratio_scenarios', 'run_all_uke_folds', 'save_first_epoch_debug_plots', 'save_val_debug_plots_every_epoch', 'seed', 'selected_ratio_name', 'setup_repeats', 'training_scenario', 'u2net_download_url', 'uke_metadata_path', 'uke_n_folds', 'uke_new_dir', 'uke_train_dir', 'uke_train_limit', 'uke_val_dir', 'uke_val_limit', 'use_patient_group_kfold_for_uke', 'wandb_entity', 'wandb_mode', 'wandb_project']
Model path: /workspaces/semedo/models/u2net.pth
SmartDoc root: /workspaces/semedo/data/external/smartdoc15
SmartDoc metadata: /workspaces/semedo/data/external/smartdoc15/frames_metadata.csv
UK

In [2]:
from pathlib import Path
import sys

finetune_dir = Path('/workspaces/semedo/src/finetune')
if str(finetune_dir) not in sys.path:
    sys.path.insert(0, str(finetune_dir))

from download_assets import download_u2net_weights, download_smartdoc15, DEFAULT_U2NET_URL

model_path = Path(config['model_path'])
root_folder = Path(config['root_folder'])
metadata_path = Path(config['opensource_metadata_path'])
u2net_url = config.get('u2net_download_url') or DEFAULT_U2NET_URL

# Downloads only run if the target files are not already present.
download_u2net_weights(model_path, u2net_url)
download_smartdoc15(root_folder, metadata_path)

u2net.pth already present at /workspaces/semedo/models/u2net.pth, skipping (use --force to redownload).
SmartDoc15 metadata already present at /workspaces/semedo/data/external/smartdoc15/frames_metadata.csv, skipping (use --force to redownload).


In [3]:
print('===== VALIDATION =====\n')

import pandas as pd

# Reload config with corrected paths
config = load_training_config()

print('Checking configured paths...')
model_path = Path(config['model_path'])
root_folder = Path(config['root_folder'])
metadata_path = Path(config['opensource_metadata_path'])
image_folder = Path(config['opensource_image_folder_path'])

print(f'  Model: {model_path}')
print(f'    exists: {model_path.is_file()}, size: {model_path.stat().st_size / 1024 / 1024:.1f} MB' if model_path.is_file() else '    MISSING')

print(f'  Data root: {root_folder}')
print(f'    exists: {root_folder.is_dir()}')

print(f'  Metadata: {metadata_path}')
print(f'    exists: {metadata_path.is_file()}')

print(f'  Image folder: {image_folder}')
print(f'    exists: {image_folder.is_dir()}')

# Count images
image_count = len(list(image_folder.glob('background*/datasheet*/frame_*.jpeg'))) if image_folder.is_dir() else 0
print(f'    images found: {image_count}')

# Load metadata
df_rows = 0
if metadata_path.is_file():
    try:
        df = pd.read_csv(metadata_path)
        df_rows = len(df)
        cols_sample = list(df.columns)[:5]
        print(f'  Metadata loaded: {df_rows} rows, columns: {cols_sample}')
    except Exception as e:
        print(f'  Metadata failed: {e}')

print()
print('All data ready for training!')
print(f'  ✓ Model checkpoint: {model_path.stat().st_size / 1024 / 1024:.1f} MB')
print(f'  ✓ SmartDoc15 frames: {image_count} images')
if df_rows > 0:
    print(f'  ✓ Metadata: {df_rows} annotations')

===== VALIDATION =====

Checking configured paths...
  Model: /workspaces/semedo/models/u2net.pth
    exists: True, size: 168.1 MB
  Data root: /workspaces/semedo/data/external/smartdoc15
    exists: True
  Metadata: /workspaces/semedo/data/external/smartdoc15/frames_metadata.csv
    exists: True
  Image folder: /workspaces/semedo/data/external/smartdoc15
    exists: True
    images found: 4391
  Metadata loaded: 24889 rows, columns: ['bg_name', 'bg_id', 'model_name', 'model_id', 'modeltype_name']

All data ready for training!
  ✓ Model checkpoint: 168.1 MB
  ✓ SmartDoc15 frames: 4391 images
  ✓ Metadata: 24889 annotations


In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    from finetuning import main
    main()
else:
    print('Training not started. Set RUN_TRAINING = True to launch it.')

UKE metadata path resolution: direct=95 basename=0 missing=0 not_found=100 ambiguous_direct=0 ambiguous_basename=0
UKE patient-group CV enabled: running all 5 folds.

===== UKE SPLIT fold1of5 | train_imgs=76 val_imgs=19 train_groups=8 val_groups=2 =====


===== SCENARIO ratio_50_50 | OS train=75 val=20 | UKE train=76 val=19 | aug_per_img=5 | split=fold1of5 =====


===== SETUP RUN 1/1 | name = uke_single_ratio_ratio_50_50_run1 | seed = 42 =====


===== REPEAT 1/1 | opensource train=75 val=20 =====

Train source sizes (after augmentation): opensource=450, uke=456
Val sizes: opensource=20 uke=19 | balanced_sampler=False


===== START TRAINING LR = 0.001 | scenario = ratio_50_50 | augs_per_img = 5 | repeat = 1/1 | setup_run = 1/1 =====



wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: ERROR Invalid API key: API key must have 40+ characters, has 36.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently lo

l0: 0.023121, l1: 0.023130, l2: 0.022568, l3: 0.022543, l4: 0.031249, l5: 0.028300, l6: 0.053143

l0: 0.029940, l1: 0.030085, l2: 0.030503, l3: 0.029435, l4: 0.037106, l5: 0.043719, l6: 0.133058

l0: 0.130081, l1: 0.130410, l2: 0.129468, l3: 0.125344, l4: 0.417316, l5: 0.378661, l6: 0.429719

l0: 0.123380, l1: 0.123934, l2: 0.125761, l3: 0.123143, l4: 0.457408, l5: 0.447749, l6: 0.083874

l0: 0.266463, l1: 0.269916, l2: 0.641089, l3: 0.428158, l4: 0.673878, l5: 0.481429, l6: 0.154602

l0: 0.246340, l1: 0.250561, l2: 0.299188, l3: 0.226740, l4: 0.441802, l5: 0.863868, l6: 0.322649

l0: 0.031272, l1: 0.031878, l2: 0.030268, l3: 0.029199, l4: 0.039351, l5: 0.043261, l6: 0.038174

l0: 0.056399, l1: 0.057946, l2: 0.050569, l3: 0.050133, l4: 0.049853, l5: 0.053518, l6: 0.052188

l0: 0.154997, l1: 0.159782, l2: 0.150749, l3: 0.137734, l4: 0.397749, l5: 0.907389, l6: 0.079396

l0: 0.035748, l1: 0.036564, l2: 0.036655, l3: 0.031096, l4: 0.065341, l5: 0.226180, l6: 0.048005

l0: 0.068300, l1: 0.

## Evaluation

Evaluates the fold checkpoint(s) just trained (plus the pretrained baseline) against the held-out UKE test set (`uke_test_dir`), then plots the per-image `selection_score` distribution. Requires `uke_metadata_path`/`uke_test_dir` to point at real (or dummy, via [create_dummy_uke_dataset.py](create_dummy_uke_dataset.py)) test data with ground truth.

In [ ]:
if RUN_TRAINING:
    from evaluate_ensemble import main as run_evaluation
    run_evaluation()

    import os
    import pandas as pd
    from plot_results import plot_violin

    training_config = load_training_config()
    checkpoint_dir = training_config.get('checkpoint_dir', '.')
    eval_output_dir = training_config.get('eval_output_dir', os.path.join(checkpoint_dir, 'test_eval'))
    per_image_csv = os.path.join(eval_output_dir, 'test_ensemble_per_image.csv')

    df = pd.read_csv(per_image_csv)
    plot_violin(df, eval_output_dir)
else:
    print('Training not started, skipping evaluation.')